# AB实验 ML增强分析

在统计分析基础上，用机器学习方法回答统计方法无法回答的问题：

1. **逻辑回归** → 处理效应可解释性
2. **随机森林** → 特征重要性排序
3. **XGBoost** → 自动捕捉特征交互
4. **HTE分析** → 新页面是否对某类用户更有效？
5. **CUPED** → 用ML预测值缩减实验方差

> 核心思路：统计方法回答「有没有效果」，ML方法回答「效果对谁有效」，两者结合形成因果+预测闭环

## 数据准备

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')
import os

DATA_PATH = './ab_data748296.xlsx'
COUNTRY_PATH = './countries237256.xlsx'
OUTPUT_DIR = './output_ml/'
RANDOM_STATE = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 数据加载与清洗（与统计版相同）
df = pd.read_excel(DATA_PATH)
countries = pd.read_excel(COUNTRY_PATH)

mask = ~(
    ((df['group'] == 'control') & (df['landing_page'] == 'new_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'old_page'))
)
df = df[mask].copy()
df = df.drop_duplicates(subset='user_id', keep='first')
df = df.merge(countries, on='user_id', how='inner')

def time_to_seconds(t):
    if hasattr(t, 'hour'):
        return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6
    try:
        parts = str(t).split(':')
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    except:
        return 0

df['time_seconds'] = df['timestamp'].apply(time_to_seconds)
df['time_min'] = df['time_seconds'] / 60

# 特征工程
df['is_treatment'] = (df['group'] == 'treatment').astype(int)
df['country_US'] = (df['country'] == 'US').astype(int)
df['country_UK'] = (df['country'] == 'UK').astype(int)
df['country_CA'] = (df['country'] == 'CA').astype(int)
df['time_min_bucket'] = pd.cut(df['time_min'], bins=[0,10,20,30,40,50,60,90], labels=False).fillna(0).astype(int)
df['quick_user'] = (df['time_seconds'] < 1800).astype(int)
df['slow_user'] = (df['time_seconds'] >= 1800).astype(int)

# 交互特征
df['treatment_x_time'] = df['is_treatment'] * df['time_seconds']
df['treatment_x_US'] = df['is_treatment'] * df['country_US']
df['treatment_x_UK'] = df['is_treatment'] * df['country_UK']
df['treatment_x_quick'] = df['is_treatment'] * df['quick_user']

print(f'数据量: {len(df):,}, 转化率: {df["converted"].mean()*100:.2f}%')

## 第一步：逻辑回归 — 处理效应可解释性

In [ ]:
feature_cols = ['is_treatment', 'time_seconds', 'country_US', 'country_UK',
                'quick_user', 'slow_user', 'time_min_bucket']
X = df[feature_cols].values.astype(float)
y = df['converted'].values

scaler = StandardScaler()
X_scaled = X.copy()
time_idx = feature_cols.index('time_seconds')
X_scaled[:, time_idx] = scaler.fit_transform(X[:, time_idx:time_idx+1]).ravel()

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, penalty=None)
lr.fit(X_scaled, y)

# Odds Ratio
coef_df = pd.DataFrame({'feature': feature_cols, 'coef': lr.coef_[0], 'odds_ratio': np.exp(lr.coef_[0])})
print('Odds Ratio：')
print(coef_df.to_string(index=False))

y_pred_proba = lr.predict_proba(X_scaled)[:, 1]
auc = roc_auc_score(y, y_pred_proba)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(lr, X_scaled, y, cv=cv, scoring='roc_auc')
print(f'\nAUC-ROC: {auc:.4f}')
print(f'5-fold CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

treatment_or = coef_df[coef_df['feature'] == 'is_treatment']['odds_ratio'].values[0]
print(f'\n>>> 处理效应OR={treatment_or:.6f}，约等于1，新页面不影响转化概率')

In [ ]:
# Odds Ratio可视化
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#E74C3C' if f == 'is_treatment' else '#4A90D9' for f in feature_cols]
ax.barh(feature_cols, coef_df['odds_ratio'].values, color=colors)
ax.axvline(x=1, color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('Odds Ratio (exp(coef))')
ax.set_title('Logistic Regression: Feature Odds Ratios\n(Red = Treatment Effect)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'logistic_odds_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

## 第二步：随机森林 — 特征重要性

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100, max_depth=10, min_samples_leaf=100,
    random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'
)
rf.fit(X, y)

importances = pd.DataFrame({'feature': feature_cols, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)
print('特征重要性排序：')
print(importances.to_string(index=False))

rf_proba = rf.predict_proba(X)[:, 1]
rf_auc = roc_auc_score(y, rf_proba)
cv_scores = cross_val_score(rf, X, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE), scoring='roc_auc', n_jobs=-1)
print(f'\nAUC-ROC: {rf_auc:.4f}')
print(f'5-fold CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

treatment_imp = importances[importances['feature'] == 'is_treatment']['importance'].values[0]
print(f'\n>>> is_treatment重要性={treatment_imp:.4f}，"是否看到新页面"对预测转化的贡献极低')

In [ ]:
# 特征重要性可视化
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#E74C3C' if f == 'is_treatment' else '#4A90D9' for f in importances['feature']]
ax.barh(importances['feature'], importances['importance'], color=colors)
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Random Forest: Feature Importance\n(Red = Treatment Effect)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 第三步：XGBoost — 自动特征交互

In [ ]:
xgb_feature_cols = ['is_treatment', 'time_seconds', 'country_US', 'country_UK',
                    'quick_user', 'slow_user', 'time_min_bucket',
                    'treatment_x_time', 'treatment_x_US', 'treatment_x_UK',
                    'treatment_x_quick']
X_xgb = df[xgb_feature_cols].values.astype(float)

xgb_clf = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    min_child_weight=100, subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, eval_metric='logloss',
    scale_pos_weight=len(y[y==0]) / len(y[y==1])
)
xgb_clf.fit(X_xgb, y)

# 特征重要性
imp_dict = xgb_clf.get_booster().get_score(importance_type='gain')
mapped_imp = {}
for k, v in imp_dict.items():
    idx = int(k.replace('f', ''))
    if idx < len(xgb_feature_cols):
        mapped_imp[xgb_feature_cols[idx]] = v

xgb_importances = pd.DataFrame({
    'feature': xgb_feature_cols,
    'gain': [mapped_imp.get(f, 0) for f in xgb_feature_cols]
}).sort_values('gain', ascending=False)
xgb_importances['gain_norm'] = xgb_importances['gain'] / xgb_importances['gain'].sum()
print('XGBoost特征重要性（Gain，归一化）：')
print(xgb_importances.to_string(index=False))

xgb_proba = xgb_clf.predict_proba(X_xgb)[:, 1]
xgb_auc = roc_auc_score(y, xgb_proba)
cv_scores = cross_val_score(xgb_clf, X_xgb, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE), scoring='roc_auc')
print(f'\nAUC-ROC: {xgb_auc:.4f}')
print(f'5-fold CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

print(f'\n>>> 交互特征增益低 -> 无显著异质性处理效应')

In [ ]:
# XGBoost特征重要性可视化
fig, ax = plt.subplots(figsize=(8, 6))
colors = []
for f in xgb_importances['feature']:
    if f == 'is_treatment': colors.append('#E74C3C')
    elif 'treatment_x' in f: colors.append('#F39C12')
    else: colors.append('#4A90D9')
ax.barh(xgb_importances['feature'], xgb_importances['gain_norm'], color=colors)
ax.set_xlabel('Feature Importance (Gain, normalized)')
ax.set_title('XGBoost: Feature Importance\n(Red=Main Effect, Orange=Interaction, Blue=Control)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'xgboost_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 第四步：HTE异质性处理效应分析

In [ ]:
# 分群体CATE估计
subgroups = {
    'US_quick': ((df['country'] == 'US') & (df['quick_user'] == 1)),
    'US_slow': ((df['country'] == 'US') & (df['slow_user'] == 1)),
    'UK_quick': ((df['country'] == 'UK') & (df['quick_user'] == 1)),
    'UK_slow': ((df['country'] == 'UK') & (df['slow_user'] == 1)),
    'CA_quick': ((df['country'] == 'CA') & (df['quick_user'] == 1)),
    'CA_slow': ((df['country'] == 'CA') & (df['slow_user'] == 1)),
}

hte_results = []
for name, mask_s in subgroups.items():
    sub = df[mask_s]
    ctrl = sub[sub['group'] == 'control']
    treat = sub[sub['group'] == 'treatment']
    if len(ctrl) < 100 or len(treat) < 100: continue
    cate = treat['converted'].mean() - ctrl['converted'].mean()
    se = np.sqrt(ctrl['converted'].var()/len(ctrl) + treat['converted'].var()/len(treat))
    hte_results.append({'subgroup': name, 'n_ctrl': len(ctrl), 'n_treat': len(treat),
                       'ctrl_rate': ctrl['converted'].mean()*100, 'treat_rate': treat['converted'].mean()*100,
                       'CATE_pp': cate*100, 'CI_lo': (cate-1.96*se)*100, 'CI_hi': (cate+1.96*se)*100, 'se_pp': se*100})

hte_df = pd.DataFrame(hte_results).sort_values('CATE_pp', ascending=True)
print(hte_df[['subgroup','n_ctrl','n_treat','CATE_pp','CI_lo','CI_hi']].to_string(index=False))
print(f'\n>>> 各子群体CATE差异小，95% CI均跨越0 -> 无异质性处理效应')

In [ ]:
# Forest Plot
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(hte_df))
ax.errorbar(hte_df['CATE_pp'], y_pos, xerr=1.96*hte_df['se_pp'],
            fmt='o', color='#4A90D9', capsize=4, capthick=1.5, markersize=6)
ax.axvline(x=0, color='red', linestyle='--', alpha=0.7)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(hte_df['subgroup'])
ax.set_xlabel('CATE (percentage points)')
ax.set_title('Heterogeneous Treatment Effect by Subgroup\n(Forest Plot with 95% CI)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'hte_forest_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## 第五步：CUPED方差缩减

In [ ]:
# 用ML预测值作为协变量
cuped_features = ['time_seconds', 'country_US', 'country_UK', 'quick_user', 'slow_user', 'time_min_bucket']
X_cuped = df[cuped_features].values.astype(float)
X_cuped_scaled = StandardScaler().fit_transform(X_cuped)

lr_cuped = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, penalty=None)
lr_cuped.fit(X_cuped_scaled, y)
y_hat = lr_cuped.predict_proba(X_cuped_scaled)[:, 1]

theta = np.cov(y, y_hat)[0, 1] / np.var(y_hat)
y_adj = y - theta * (y_hat - y_hat.mean())

var_original = y.var()
var_adjusted = y_adj.var()
variance_reduction = (1 - var_adjusted / var_original) * 100

print(f'原始方差: {var_original:.6f}')
print(f'调整后方差: {var_adjusted:.6f}')
print(f'方差缩减: {variance_reduction:.2f}%')

# CUPED调整后的z检验
ctrl_mask = df['group'] == 'control'
treat_mask = df['group'] == 'treatment'

def two_sample_z(y1, y2):
    n1, n2 = len(y1), len(y2)
    m1, m2 = y1.mean(), y2.mean()
    se = np.sqrt(y1.var()/n1 + y2.var()/n2)
    z = (m1 - m2) / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p, se

z_orig, p_orig, se_orig = two_sample_z(y[ctrl_mask.values], y[treat_mask.values])
z_adj, p_adj, se_adj = two_sample_z(y_adj[ctrl_mask.values], y_adj[treat_mask.values])
se_reduction = (1 - se_adj/se_orig) * 100
n_equiv_boost = (se_orig / se_adj) ** 2

print(f'\n原始: z={z_orig:.4f}, p={p_orig:.4f}, SE={se_orig:.6f}')
print(f'CUPED: z={z_adj:.4f}, p={p_adj:.4f}, SE={se_adj:.6f}')
print(f'SE缩减: {se_reduction:.2f}%')
print(f'等效样本量增加: {(n_equiv_boost-1)*100:.1f}%')

In [ ]:
# CUPED可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(['Original', 'CUPED'], [var_original, var_adjusted], color=['#4A90D9', '#E74C3C'], width=0.5)
axes[0].set_ylabel('Variance')
axes[0].set_title(f'CUPED Variance Reduction: {variance_reduction:.1f}%')

diff_orig = y[ctrl_mask.values].mean() - y[treat_mask.values].mean()
diff_adj = y_adj[ctrl_mask.values].mean() - y_adj[treat_mask.values].mean()
axes[1].errorbar([0, 1], [diff_orig*100, diff_adj*100], yerr=[1.96*se_orig*100, 1.96*se_adj*100],
                  fmt='o', color='#4A90D9', capsize=8, capthick=2, markersize=8)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.7)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Original', 'CUPED'])
axes[1].set_ylabel('Difference (pp)')
axes[1].set_title('95% CI Comparison: Original vs CUPED')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'cuped_variance_reduction.png', dpi=150, bbox_inches='tight')
plt.show()

## 第六步：模型对比汇总

In [ ]:
model_names = ['Logistic Regression', 'Random Forest', 'XGBoost']
model_probas = [y_pred_proba, rf_proba, xgb_proba]
model_aucs = [roc_auc_score(y, p) for p in model_probas]

for name, auc in zip(model_names, model_aucs):
    print(f'{name:>22}: AUC = {auc:.4f}')

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#4A90D9', '#27AE60', '#E74C3C']
bars = ax.bar(['Logistic\nRegression', 'Random\nForest', 'XGBoost'], model_aucs, color=colors, width=0.5)
ax.set_ylabel('AUC-ROC')
ax.set_title('Model Comparison: AUC-ROC')
for bar, auc_val in zip(bars, model_aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001, f'{auc_val:.4f}', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'model_comparison_auc.png', dpi=150, bbox_inches='tight')
plt.show()

## 面试核心结论

| 方法 | 核心发现 |
|------|----------|
| 逻辑回归 | 处理效应OR≈1，新页面不影响转化 |
| 随机森林 | is_treatment重要性极低，访问耗时是关键驱动因素 |
| XGBoost | 交互项增益低，无显著异质性处理效应 |
| HTE分析 | 分群体CATE均不显著，新页面不对任何特定群体更有效 |
| CUPED | 用ML预测值缩减方差，展示工业界前沿方法 |

> **一句话总结**：统计方法回答「有没有效果」→ 没有；ML方法回答「效果对谁有效」→ 对谁都没有。两个角度一致，结论更可信。